In [1]:
%load_ext autoreload
%autoreload 2
%pdb on # clickable err traceback



from __future__ import absolute_import, division, print_function
import torch
from trainer_endoda3 import Trainer
from options_endoda3 import MonodepthOptions


Incorrect argument. Use on/1, off/0, or nothing for a toggle.


/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [2]:
# Minimal options for testing
options = MonodepthOptions()
args = [
    '--batch_size', '8',
    '--batch_size', '2',
    '--batch_size', '1',
    '--num_workers', '0',
    '--of_samples',
    '--of_samples_num', '1',
    '--of_samples_num', '10',
    '--frame_ids', '0', '-1', '1',
    '--train_frame_ids', '0', '-5', '5',
    '--train_frame_ids', '0', '-1', '1',
    '--dataset', 'endovis',
    '--data_path', '/mnt/cluster/workspaces/jinjingxu/SCARED_Images_Resized/',
    '--log_dir', '/tmp/endoda_debug',
    '--log_dir', '/mnt/cluster/workspaces/jinjingxu/tmp/',
    '--compute_depth_metrics',
    '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-all-wowrapper.yaml',
    '--pose_model_type', 'da3_internal',
    '--k_model_type', 'da3_internal',
    '--of_supervised_with_which', 'inputs_color',
    '--learn_intrinsics',
    '--train_data_file', 'test_files.txt',
    '--train_data_file', 'test_files_sequence1_val.txt',
    '--train_data_file', 'train_files.txt',
    '--val_data_file', 'test_files.txt test_files_sequence1_val.txt  test_files_sequence2_val.txt ',
    '--val_data_file', 'test_files.txt',
    '--val_data_file', 'test_files_sequence1_val.txt',
    '--da3_depth_regression_target', 'disp',
    '--da3_depth_regression_target', 'disp',
    '--da3_depth_regression_target', 'depth2disp',
    '--da3_depth_regression_target', 'depth2disp_v2',
    '--da3_depth_regression_target', 'depth2disp_v3',
    '--depth_model_type', 'endodac',
    '--depth_model_type', 'depthanything3',
    '--k_model_type', 'mlp_with_pn_bottleneck_ipt',
    '--k_model_type', 'da3_internal',
    '--warm_up_step', '5000',
    '--pose_model_type', 'separate_resnet',
    '--pretrained_path', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/weights/depthanything',
    '--pretrained_path', 'depth-anything/da3-base',
    '--af_model_type', 'adjust_net',
    '--of_model_type', 'raft',
    # '--depth_model_type', 'endodac',

# --depth_model_type endodac --da3_depth_regression_target disp --k_model_type mlp_with_pn_bottleneck_ipt --warm_up_step 5000 --pretrained_path /mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/weights/depthanything \


    # '--enable_seq_inputs',
    # '--of_model_type', 'raft',
    # '--use_raft_multi_iters',
    # '--raft_trainable_modules', 'convnormrelu layer1 layer2_0 ',
    # '--learn_intrinsics',
    # '--use_perframe_gt_K',
 
    # '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-depth-wowrapper.yaml'
]
opts = options.parse_notebook(args)


# Initialize trainer
trainer = Trainer(opts)
print(f"Trainer initialized on {trainer.device}")


Loading depth model setting from config: /mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-all-wowrapper.yaml
[INFO ] using MLP layer as FFN

Loading pretrained weights from depth-anything/da3-base
[INFO ] using MLP layer as FFN
***disable_load_pretrained_weights_cam_dec:  ['cam_dec.fc_qvec', 'cam_dec.fc_t']
  Excluded 4 keys from pretrained weights based on disabled modules: ['cam_dec.fc_qvec', 'cam_dec.fc_t']

depth_model state dict info:
  Missing keys: 100
    Key prefixes (up to 2 levels): ['backbone.blocks', 'cam_dec.fc_qvec', 'cam_dec.fc_t', 'head.conv_depth_aux_1', 'head.conv_depth_aux_2', 'head.conv_depth_aux_3', 'head.conv_depth_aux_4', 'head.conv_depth_main_1', 'head.conv_depth_main_2', 'head.conv_depth_main_3', 'head.conv_depth_main_4']
  Unexpected keys: 0
Successfully loaded pretrained weights from depth-anything/da3-base for depth net.

Training model named:
   0104_0236
Models and tensorboard events files are saved to:
   /mnt/cluster/works

/mnt/cluster/environments/jinjingxu/pkg/envs/cu12/lib/python3.11/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647348947/work/aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [3]:
# # loop over trainer.val_loader to check the collate_fn
# for i, val_inputs in enumerate(trainer.val_loader):
#     print("Batch", i)

In [4]:
# Get sample batch
trainer.step = 0
trainer.set_train()
trainer.current_frame_ids = trainer.train_frame_ids  # Set context for training
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)
val_iter = iter(trainer.val_loader)
val_inputs = next(val_iter)

# Forward pass
outputs, losses = trainer.process_batch(inputs)

print("Forward pass completed!")
print(f"Output keys: {len(outputs)} keys")
print(f"Loss: {losses['loss'].item():.6f}")
print(f"Loss components: {list(losses.keys())}")


Resizing input from 256x320 to 224x280


/mnt/cluster/environments/jinjingxu/pkg/envs/cu12/lib/python3.11/site-packages/torch/nn/functional.py:4902: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  warnings.warn(


Resizing input from 256x320 to 224x280
Resizing input from 256x320 to 224x280
Forward pass completed!
Output keys: 156 keys
Loss: 0.195586
Loss components: ['loss_reprojection/0', 'loss_explict_geo/0', 'loss_transform/0', 'loss_cvt/0', 'loss_smooth/0', 'loss/0', 'loss_reprojection/1', 'loss_explict_geo/1', 'loss_transform/1', 'loss_cvt/1', 'loss_smooth/1', 'loss/1', 'loss_reprojection/2', 'loss_explict_geo/2', 'loss_transform/2', 'loss_cvt/2', 'loss_smooth/2', 'loss/2', 'loss_reprojection/3', 'loss_explict_geo/3', 'loss_transform/3', 'loss_cvt/3', 'loss_smooth/3', 'loss/3', 'loss']


In [5]:
# Compute losses explicitly
# Ensure current_frame_ids is set for training context
trainer.current_frame_ids = trainer.train_frame_ids
losses = trainer.compute_losses(inputs, outputs)

print("Loss computation:")
for key, val in losses.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.item():.6f}")
    else:
        print(f"  {key}: {val}")

# compute losses_0 explicitly
losses_0 = trainer.compute_losses_0(inputs, outputs)

print("Loss computation_0:")
for key, val in losses_0.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.item():.6f}")
    else:
        print(f"  {key}: {val}")

Loss computation:
  loss_reprojection/0: 0.200770
  loss_explict_geo/0: 0.821146
  loss_transform/0: 0.067729
  loss_cvt/0: 0.001148
  loss_smooth/0: 0.006658
  loss/0: 0.201465
  loss_reprojection/1: 0.192822
  loss_explict_geo/1: 0.243585
  loss_transform/1: 0.046363
  loss_cvt/1: 0.005454
  loss_smooth/1: 0.008762
  loss/1: 0.193349
  loss_reprojection/2: 0.196234
  loss_explict_geo/2: 0.064538
  loss_transform/2: 0.046889
  loss_cvt/2: 0.001175
  loss_smooth/2: 0.014003
  loss/2: 0.196729
  loss_reprojection/3: 0.190445
  loss_explict_geo/3: 0.015611
  loss_transform/3: 0.033854
  loss_cvt/3: 0.000334
  loss_smooth/3: 0.014768
  loss/3: 0.190802
  loss: 0.195586
Loss computation_0:
  loss/0: 0.145551
  loss/1: 0.153944
  loss/2: 0.167681
  loss/3: 0.178575
  loss: 0.161438


In [6]:
# optional: obtain various image from outputs and save as one row of images of all image
from utils import img_gen
for key in outputs.keys():
    outputs[key] = outputs[key].detach()
    # print(f"  {key}: shape={outputs[key].shape}")
# compute the depth_err metrics    
from utils.metrics import compute_depth_metrics
metrics = compute_depth_metrics(inputs, outputs)
merged_dict = {**inputs, **outputs, **metrics}
img_gen(
    merged_dict=merged_dict,
    image_keys_row1=[
    ("color", 0, 0), 
    ("color_aug", 0, 0), 
    ("color", -1, 0), 
    ("color_aug", -1, 0), 
    # ("disp", 0), 
    ("depth", 0, 0), 
    ("depth_gt", 0, 0), 
    ("occu_mask_backward", 0, -1), 
    ("occu_mask_backward", 1, -1), 
    ("occu_mask_backward", 3, -1), 

    ("pose_flow", "high", -1, 3), 
    ("position", "high", 3, -1), 
    
    # "depth_err",
    # "depth_err",
    ],
    # image_keys_row2=[
    #     ("color_aug", 0, 0),  # Target
    #     ("color_warp", 0, -1),  # Raw warped
    #     ("paba_color_warp", 0, -1),  # Aligned
    #     ("paba_alpha", 0, -1),  # Alpha map
    #     "color_warp_err_before",
    #     "color_warp_err_after_afstyle_color_warp",
    # ],
    save_path="output_grid.png",
    sample_idx=0
)

Saved image grid to output_grid.png


In [7]:
# Set context for validation
trainer.current_frame_ids = trainer.val_frame_ids
val_outputs, _ = trainer.process_batch_val(val_inputs)
val_losses = trainer.compute_losses_val(val_inputs, val_outputs)

Resizing input from 256x320 to 224x280


Resizing input from 256x320 to 224x280
Resizing input from 256x320 to 224x280


In [8]:
# Get validation batch (has GT depth and poses)
trainer.set_eval()
trainer.current_frame_ids = trainer.val_frame_ids  # Set context for validation
val_iter = iter(trainer.val_loader)
val_inputs = next(val_iter)

# for flag in [True, False]:
    # trainer.replace_with_gt_rel_rotation = flag
    # print('replace_with_gt_rel_rotation: ', flag)

# Forward pass
with torch.no_grad():
    val_outputs, val_losses = trainer.process_batch_val(val_inputs)

# Compute depth metrics
from utils.metrics import compute_depth_metrics, compute_pose_metrics

depth_metrics = compute_depth_metrics(val_inputs, val_outputs)
# Use current_frame_ids which is already set to val_frame_ids
pose_metrics = compute_pose_metrics(val_inputs, val_outputs, trainer.current_frame_ids)

print("Depth Metrics:")
if depth_metrics:
    for key, val in depth_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No depth metrics (GT depth not available)")

print("\nPose Metrics:")
print(pose_metrics)
if pose_metrics:
    for key, val in pose_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No pose metrics (GT poses not available)")


Resizing input from 256x320 to 224x280


Resizing input from 256x320 to 224x280
Resizing input from 256x320 to 224x280
Depth Metrics:
  No depth metrics (GT depth not available)

Pose Metrics:
{'pose_trans_err_ang_deg': 90.84914016723633, 'pose_rot_err_deg': 0.11088304594159126, 'pose_pred_rel_trans_scale': 0.0001114373626478482, 'pose_pred_f0_depth_scale': 0.9699687957763672}
  pose_trans_err_ang_deg: 90.849140
  pose_rot_err_deg: 0.110883
  pose_pred_rel_trans_scale: 0.000111
  pose_pred_f0_depth_scale: 0.969969


In [9]:
# Full training step
trainer.set_train()
trainer.current_frame_ids = trainer.train_frame_ids  # Set context for training
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)

# Forward
outputs, losses = trainer.process_batch(inputs)

# Backward
trainer.model_optimizer.zero_grad()
losses["loss"].backward()
trainer.model_optimizer.step()

print(f"Training step completed! Loss: {losses['loss'].item():.6f}")


Resizing input from 256x320 to 224x280
Resizing input from 256x320 to 224x280
Resizing input from 256x320 to 224x280
Training step completed! Loss: 0.194688
